# 損傷画像読解・補修優先度スコアリング MVP デモ

このノートブックでは、橋梁損傷画像から自動的にテキスト説明を生成し、構造化してスコアリングするパイプラインをデモします。

## パイプライン全体像
```
画像入力 → 前処理 → Granite-Vision → JSON構造化 → スコアリング → 優先度出力
```

## 1. セットアップ

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# プロジェクトルートをパスに追加
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

print(f"プロジェクトルート: {project_root}")
print(f"Python: {sys.version}")

## 2. データ確認

In [ ]:
# データディレクトリ
data_dir = project_root / 'data' / 'images_human_inspect_n254'
image_files = sorted(data_dir.glob('*.png'))

print(f"画像数: {len(image_files)}枚")
print(f"\nサンプル画像:")
for i, img_path in enumerate(image_files[:5], 1):
    print(f"  {i}. {img_path.name}")

In [ ]:
# サンプル画像表示
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, img_path in enumerate(image_files[:6]):
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(img_path.name, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. 前処理テスト

In [ ]:
from preprocessing.image_preprocessor import ImagePreprocessor, PreprocessConfig

# 前処理設定
preprocess_config = PreprocessConfig(
    max_width=1024,
    max_height=1024,
    denoise_enabled=True,
    denoise_strength=5,
    contrast_enabled=True,
    clip_limit=2.0
)

preprocessor = ImagePreprocessor(preprocess_config)
print("前処理モジュール初期化完了")

In [ ]:
# サンプル画像で前処理テスト
sample_img = image_files[0]
print(f"処理画像: {sample_img.name}")

# 前処理実行
preprocessed_dir = project_root / 'data' / 'preprocessed'
preprocessed_dir.mkdir(parents=True, exist_ok=True)

preprocessed_img = preprocessor.preprocess_file(
    sample_img,
    preprocessed_dir / sample_img.name
)

# 比較表示
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

original = Image.open(sample_img)
axes[0].imshow(original)
axes[0].set_title('元画像')
axes[0].axis('off')

processed = Image.open(preprocessed_dir / sample_img.name)
axes[1].imshow(processed)
axes[1].set_title('前処理後')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Granite-Vision分析テスト

In [ ]:
from vision.granite_vision import GraniteVisionAnalyzer, VisionConfig

# Vision設定
vision_config = VisionConfig(
    model_name='ibm-granite/granite-vision-3b',
    device='cuda',
    max_new_tokens=300,
    temperature=0.3
)

print("Granite-Visionモデル読み込み中...")
vision_analyzer = GraniteVisionAnalyzer(vision_config)
print("Vision分析モジュール初期化完了")

In [ ]:
# サンプル画像で分析テスト
print(f"分析画像: {sample_img.name}")

description = vision_analyzer.analyze_image(sample_img)

print(f"\n=== 損傷説明 ===")
print(description)

In [ ]:
# 複数画像で分析（3枚）
print("複数画像分析中...\n")

test_images = image_files[:3]
descriptions = []

for img_path in test_images:
    desc = vision_analyzer.analyze_image(img_path)
    descriptions.append(desc)
    print(f"--- {img_path.name} ---")
    print(desc[:150] + '...')
    print()

## 5. JSON構造化テスト

In [ ]:
from structuring.json_structurer import JSONStructurer, StructuringConfig

# 構造化設定
structuring_config = StructuringConfig(
    model_name='tokyotech-llm/Swallow-7b-instruct-v0.1',
    device='cuda',
    max_new_tokens=500,
    temperature=0.1
)

print("JSON構造化モデル読み込み中...")
structurer = JSONStructurer(structuring_config)
print("JSON構造化モジュール初期化完了")

In [ ]:
# サンプル説明文を構造化
structure = structurer.structure_text(description)

print("=== 構造化結果 ===")
print(json.dumps(structure.to_dict(), ensure_ascii=False, indent=2))

## 6. スコアリングテスト

In [ ]:
from scoring.priority_scorer import PriorityScorer, ScoringConfig

# スコアリング設定
scoring_config = ScoringConfig(
    rules_file=str(project_root / 'models' / 'scoring_rules.yaml'),
    weight_damage_type=0.35,
    weight_severity=0.40,
    weight_location=0.15,
    weight_risk=0.10,
    gam_enabled=False
)

scorer = PriorityScorer(scoring_config)
print("スコアリングモジュール初期化完了")

In [ ]:
# スコアリング実行
score = scorer.calculate_score(
    damage_type=structure.damage_type,
    severity=structure.severity,
    location=structure.location,
    risk=structure.risk
)

print("=== スコアリング結果 ===")
print(json.dumps(score.to_dict(), ensure_ascii=False, indent=2))

## 7. エンドツーエンドパイプライン実行

In [ ]:
from pipeline.end_to_end import DamageAnalysisPipeline

# パイプライン初期化
config_path = project_root / 'config.yaml'
pipeline = DamageAnalysisPipeline(str(config_path))

In [ ]:
# 単一画像をエンドツーエンド処理
test_img = image_files[0]
print(f"処理画像: {test_img.name}\n")

result = pipeline.process_image(test_img)

print(f"\n=== 処理結果 ===")
print(f"ステータス: {result.status}")
print(f"処理時間: {result.processing_time:.2f}秒")
print(f"\n損傷説明:")
print(result.description[:200] + '...')
print(f"\n構造化データ:")
print(json.dumps(result.structure, ensure_ascii=False, indent=2))
print(f"\nスコア:")
print(f"  生スコア: {result.score['raw_score']:.3f}")
print(f"  優先度レベル: {result.score['priority_level']}")
print(f"  説明: {result.score['priority_description']}")

In [ ]:
# 複数画像を一括処理（10枚）
print("一括処理を開始...\n")

results = pipeline.process_batch(
    input_dir=data_dir,
    pattern='*.png',
    limit=10
)

print(f"\n処理完了: {len(results)}枚")

In [ ]:
# 結果をDataFrameに変換
rows = []
for r in results:
    if r.status == 'success':
        rows.append({
            '画像': r.image_name,
            '損傷種別': r.structure['damage_type'],
            '重症度': r.structure['severity'],
            '位置': r.structure['location'],
            'リスク': r.structure['risk'],
            'スコア': r.score['raw_score'],
            '優先度': r.score['priority_level'],
            '処理時間': r.processing_time
        })

df = pd.DataFrame(rows)
df

In [ ]:
# 結果の可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 優先度分布
priority_counts = df['優先度'].value_counts().sort_index(ascending=False)
axes[0].bar(priority_counts.index, priority_counts.values, color='steelblue')
axes[0].set_xlabel('優先度レベル')
axes[0].set_ylabel('件数')
axes[0].set_title('優先度分布')
axes[0].grid(axis='y', alpha=0.3)

# 損傷種別分布
damage_counts = df['損傷種別'].value_counts()
axes[1].barh(damage_counts.index, damage_counts.values, color='coral')
axes[1].set_xlabel('件数')
axes[1].set_title('損傷種別分布')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 結果保存

In [ ]:
# CSV保存
output_dir = project_root / 'data' / 'outputs'
pipeline.save_results(results, output_dir, 'demo_results.csv')

print(f"結果を保存しました: {output_dir}")

## 9. まとめ

このノートブックでは、以下を実行しました：

1. ✅ 画像前処理（denoise, resize, contrast調整）
2. ✅ Granite-Visionによる損傷説明生成
3. ✅ JSON構造化（損傷種別、重症度、位置、リスク）
4. ✅ ルールベーススコアリング
5. ✅ エンドツーエンドパイプライン実行
6. ✅ 結果の可視化と保存

### 次のステップ

- 全254枚の画像を処理
- GAMモデルで補正精度向上
- 人手ラベルとの比較評価
- UI開発